# Analisi emotiva di una discussione completa

Questo notebook analizza una discussione Reddit (post + tutti i suoi commenti) e ne calcola:
1. **Emozione del post** — con il metodo finale (ELIta α=0.5 + corpus_mean ItEm)
2. **Emozione di ogni commento** — stessa pipeline
3. **Emozione complessiva della discussione** — media dei vettori normalizzati

Di default viene selezionato il post con più commenti nel corpus.

## Import e configurazione

In [1]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from Fase3.support import (
    BASIC_EMOTIONS, EMOTION_COLORS, POS_FILTER,
    load_corpus, load_recalc, compute_mu_e,
    score_single_document, normalizza, emozione_dominante,
    plot_emotion_bars, plot_radar_single, plot_radar_comparison,
)

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')

print('Configurazione caricata.')


Configurazione caricata.


## Caricamento dati e calcolo μ_e

In [2]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
df_elita = load_recalc(ALPHA_05_CSV)

# Dizionario doc_id → lista lemmi filtrati (ADJ/NOUN/VERB) — usato per scoring per-documento
df_f     = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
tok_dict = df_f.groupby('doc_id')['lemma'].apply(list).to_dict()
elita_idx = set(df_elita.index)

# μ_e su tutto il corpus (necessario per la corpus_mean normalisation)
mu_e = compute_mu_e(df_corpus, df_tokens, df_elita)

print(f'Corpus: {len(df_corpus)} documenti | Token: {len(df_tokens)}')
print(f'  post: {(df_corpus["type"]=="post").sum()} | commenti: {(df_corpus["type"]=="comment").sum()}')
print()
print('Medie corpus μ_e (ELIta α=0.5):')
for e in BASIC_EMOTIONS:
    print('  {:<15s}: {:.4f}'.format(e, mu_e[e]))


Corpus: 2360 documenti | Token: 80463
  post: 200 | commenti: 2160

Medie corpus μ_e (ELIta α=0.5):
  gioia          : 4.6511
  tristezza      : 3.3851
  rabbia         : 3.3335
  paura          : 3.7742
  disgusto       : 2.4609
  fiducia        : 4.9203
  sorpresa       : 4.6514
  aspettativa    : 5.6918


## Selezione del post

Di default: il post con più commenti nel corpus.  
Puoi cambiare impostando `POST_ID = 'id_del_post'` manualmente.

In [3]:
POST_ID = None   # es. '1idmjsb' — lascia None per il post con più commenti

df_comments_only = df_corpus[df_corpus['type'] == 'comment']
df_posts_only    = df_corpus[df_corpus['type'] == 'post']

if POST_ID is None:
    # Post con più commenti nel corpus
    post_id_max = df_comments_only['post_id'].value_counts().idxmax()
    POST_ID = post_id_max

# Riga del post
post_row = df_posts_only[df_posts_only['doc_id'] == POST_ID].iloc[0]

# Commenti del post
df_disc_comments = df_comments_only[df_comments_only['post_id'] == POST_ID].reset_index(drop=True)

print(f'Post selezionato : {POST_ID}')
print(f'Autore           : {post_row["author"]}')
print(f'Score Reddit     : {post_row["score"]}')
print(f'Commenti nel corpus: {len(df_disc_comments)}')
print()
print('Testo del post:')
print('=' * 70)
print(post_row['text'])
print('=' * 70)

Post selezionato : 1kwlwc6
Autore           : Heavy-Ad1398
Score Reddit     : 31
Commenti nel corpus: 25

Testo del post:
Quanto odio quei siti di notizie...
...in cui ti chiedono di accettare 200 biscotti 🍪 🍪 per poter visualizzare l'articolo e non appena li accetti ti comunicano che comunque è a pagamento quindi sticazzi, non lo puoi leggere. 

Ma è legale sta cosa? Appena provato a leggere un articolo di quifinanza. Ho scandagliato le impostazioni e li ho eliminati, col cazzo che ve li lascio, STRONZI!


## Analisi emotiva del post

In [4]:
sc_post_raw, found_post = score_single_document(POST_ID, tok_dict, elita_idx, df_elita)
sc_post_norm = normalizza(sc_post_raw, mu_e)
emo_post     = emozione_dominante(sc_post_norm)

print(f'Token usati per il post: {found_post}')
print()
print('{:<15s} {:>10s} {:>10s} {:>10s}'.format('Emozione', 'S_e (raw)', 'μ_e', 'S_e_norm'))
print('-' * 50)
for e in BASIC_EMOTIONS:
    marker = ' ◄' if e == emo_post else ''
    print('{:<15s} {:>10.3f} {:>10.3f} {:>10.3f}{}'.format(
        e, sc_post_raw[e], mu_e[e], sc_post_norm[e], marker))
print()
print(f'=> Emozione del POST: {emo_post.upper()}')

plot_radar_single(
    sc_post_norm, emo_post,
    title=f'Profilo emotivo del post {POST_ID} — emozione dominante: {emo_post.upper()}',
    height=450,
).show()


Token usati per il post: 19

Emozione         S_e (raw)        μ_e   S_e_norm
--------------------------------------------------
gioia                9.252      4.651      1.989
tristezza            5.559      3.385      1.642
rabbia               5.623      3.334      1.687
paura                6.145      3.774      1.628
disgusto             4.656      2.461      1.892
fiducia              9.560      4.920      1.943
sorpresa             9.517      4.651      2.046 ◄
aspettativa         11.329      5.692      1.990

=> Emozione del POST: SORPRESA


## Analisi emotiva di ogni commento

In [5]:
comment_results = []
for _, row in df_disc_comments.iterrows():
    sc_raw, found = score_single_document(row['doc_id'], tok_dict, elita_idx, df_elita)
    sc_norm = normalizza(sc_raw, mu_e)
    emo     = emozione_dominante(sc_norm)
    comment_results.append({
        'doc_id'         : row['doc_id'],
        'rank_by_score'  : row.get('rank_by_score', ''),
        'score_reddit'   : row['score'],
        'n_token_matched': found,
        'emozione'       : emo,
        'text_preview'   : str(row['text'])[:80] + '...' if len(str(row['text'])) > 80 else str(row['text']),
        **{f'norm_{e}': sc_norm[e] for e in BASIC_EMOTIONS}
    })

df_results = pd.DataFrame(comment_results)

print(f'Commenti analizzati: {len(df_results)}')
print(f'Con almeno 1 token in ELIta: {(df_results["n_token_matched"] > 0).sum()}')
print()
print('Distribuzione emozioni dominanti:')
counts = df_results['emozione'].value_counts()
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts.get(e, 0)
    if n > 0:
        print(f'  {e:<15s} {n:>3d}  {"█" * n}')
print()

display(df_results[['rank_by_score', 'score_reddit', 'emozione', 'n_token_matched', 'text_preview']]
        .sort_values('rank_by_score').reset_index(drop=True))


Commenti analizzati: 25
Con almeno 1 token in ELIta: 21

Distribuzione emozioni dominanti:
  gioia             5  █████
  tristezza         1  █
  rabbia            2  ██
  paura             2  ██
  disgusto          3  ███
  fiducia           3  ███
  sorpresa          3  ███
  aspettativa       2  ██
  neutrale          4  ████



,rank_by_score,score_reddit,emozione,n_token_matched,text_preview
0,1.0,16,neutrale,0,"Easy, non li apro."
1,2.0,11,tristezza,5,Più facile: usa ublok origin o Brave Browser e...
2,3.0,6,gioia,1,Mica lo sai prima che non te li mostrano!
3,4.0,4,paura,23,Lo fanno perché il mio commento può essere int...
4,5.0,4,disgusto,5,I cookie ormai li rifiuto sempre e comunque \n...
5,6.0,4,aspettativa,1,No puoi proprio bloccarli dal principio sia co...
6,7.0,4,sorpresa,8,"Apro in modalità in incognito, accetto i cooki..."
7,8.0,3,rabbia,4,Non capisco perché ti downvotano...a giudicare...
8,9.0,3,neutrale,0,🧐 e come ti informi?
9,10.0,3,disgusto,1,"Stronzi, hai detto bene."


In [6]:
plot_emotion_bars(
    counts, len(df_results),
    title=f'Distribuzione emozioni dominanti — commenti del post {POST_ID}',
    height=400,
).show()

# Heatmap: ogni riga = un commento, colonne = emozioni normalizzate
norm_cols = [f'norm_{e}' for e in BASIC_EMOTIONS]
df_heat = df_results[norm_cols].copy()
df_heat.columns = BASIC_EMOTIONS

import plotly.express as px
px.imshow(
    df_heat.T,
    labels=dict(x='Commento (rank)', y='Emozione', color='Score norm.'),
    x=[f'#{i+1}' for i in range(len(df_results))],
    color_continuous_scale='YlOrRd',
    title=f'Heatmap score normalizzati per commento — post {POST_ID}',
    height=400,
).show()

## Emozione complessiva della discussione

L'emozione della **discussione** è calcolata come media dei vettori normalizzati di tutti i documenti (post + commenti).  
Mostriamo anche il confronto tra emozione del post e emozione media dei commenti.

In [7]:
# Vettori normalizzati: post + commenti
all_norm_vecs = [sc_post_norm] + [
    {e: row[f'norm_{e}'] for e in BASIC_EMOTIONS}
    for _, row in df_results.iterrows()
]
df_norm_vecs = pd.DataFrame(all_norm_vecs)

# Media commenti (senza post)
sc_commenti_mean = df_norm_vecs.iloc[1:][BASIC_EMOTIONS].mean().to_dict()
emo_commenti     = emozione_dominante(sc_commenti_mean)

# Media intera discussione (post + commenti)
sc_discussione   = df_norm_vecs[BASIC_EMOTIONS].mean().to_dict()
emo_discussione  = emozione_dominante(sc_discussione)

print('=' * 60)
print('RIEPILOGO EMOTIVO DELLA DISCUSSIONE')
print('=' * 60)
print(f'  Emozione del POST            : {emo_post.upper()}')
print(f'  Emozione media COMMENTI      : {emo_commenti.upper()}')
print(f'  Emozione DISCUSSIONE totale  : {emo_discussione.upper()}')
print()

# Tabella comparativa
print('{:<15s} {:>12s} {:>14s} {:>14s}'.format('Emozione', 'POST', 'Media comm.', 'Discussione'))
print('-' * 60)
for e in BASIC_EMOTIONS:
    markers = []
    if e == emo_post:        markers.append('POST')
    if e == emo_commenti:    markers.append('COMM')
    if e == emo_discussione: markers.append('DISC')
    tag = ' ◄ ' + '+'.join(markers) if markers else ''
    print('{:<15s} {:>12.3f} {:>14.3f} {:>14.3f}{}'.format(
        e, sc_post_norm[e], sc_commenti_mean[e], sc_discussione[e], tag))

RIEPILOGO EMOTIVO DELLA DISCUSSIONE
  Emozione del POST            : SORPRESA
  Emozione media COMMENTI      : DISGUSTO
  Emozione DISCUSSIONE totale  : DISGUSTO

Emozione                POST    Media comm.    Discussione
------------------------------------------------------------
gioia                  1.989          0.587          0.641
tristezza              1.642          0.621          0.660
rabbia                 1.687          0.645          0.685
paura                  1.628          0.634          0.672
disgusto               1.892          0.675          0.722 ◄ COMM+DISC
fiducia                1.943          0.616          0.667
sorpresa               2.046          0.648          0.702 ◄ POST
aspettativa            1.990          0.614          0.667


In [8]:
plot_radar_comparison(
    traces_data=[
        ('Post',               sc_post_norm,      EMOTION_COLORS.get(emo_post, '#999'), 0.5, 'solid'),
        ('Media commenti',     sc_commenti_mean,  '#1E88E5',                            0.4, 'solid'),
        ('Discussione totale', sc_discussione,    '#43A047',                            0.3, 'dot'),
    ],
    title=f'Confronto emotivo: Post vs Commenti vs Discussione — {POST_ID}',
).show()

# Grafico a barre affiancate
fig2 = go.Figure()
for label, sc, color in [
    ('Post',               sc_post_norm,      '#FB8C00'),
    ('Media commenti',     sc_commenti_mean,  '#1E88E5'),
    ('Discussione totale', sc_discussione,    '#43A047'),
]:
    fig2.add_trace(go.Bar(
        name=label, x=BASIC_EMOTIONS,
        y=[sc[e] for e in BASIC_EMOTIONS],
        marker_color=color, opacity=0.85,
    ))
fig2.update_layout(
    barmode='group',
    title=f'Score normalizzati: Post vs Commenti vs Discussione — {POST_ID}',
    xaxis_title='Emozione', yaxis_title='Score normalizzato', height=450,
)
fig2.show()


## Accordo/disaccordo tra post e commenti

Verifichiamo se l'emozione del post coincide con quella prevalente nei commenti.

In [9]:
n_concordi   = (df_results['emozione'] == emo_post).sum()
n_discordi   = len(df_results) - n_concordi
perc_concordi = n_concordi / len(df_results) * 100 if len(df_results) > 0 else 0

print(f'Emozione del post              : {emo_post.upper()}')
print(f'Commenti con stessa emozione   : {n_concordi}/{len(df_results)} ({perc_concordi:.1f}%)')
print(f'Commenti con emozione diversa  : {n_discordi}/{len(df_results)} ({100-perc_concordi:.1f}%)')
print()

# Distribuzione emozioni nei commenti che differiscono dal post
df_discordi = df_results[df_results['emozione'] != emo_post]
if len(df_discordi) > 0:
    print('Emozioni nei commenti discordi:')
    for e, cnt in df_discordi['emozione'].value_counts().items():
        print(f'  {e:<15s}: {cnt}')

# Grafico a torta: accordo vs disaccordo
fig = go.Figure(go.Pie(
    labels=['Accordo con post', 'Emozione diversa'],
    values=[n_concordi, n_discordi],
    marker_colors=[EMOTION_COLORS.get(emo_post, '#FB8C00'), '#BDBDBD'],
    hole=0.4
))
fig.update_layout(
    title=f'Accordo emotivo commenti con post ({emo_post.upper()}) — {POST_ID}',
    height=400
)
fig.show()

Emozione del post              : SORPRESA
Commenti con stessa emozione   : 3/25 (12.0%)
Commenti con emozione diversa  : 22/25 (88.0%)

Emozioni nei commenti discordi:
  gioia          : 5
  neutrale       : 4
  disgusto       : 3
  fiducia        : 3
  paura          : 2
  aspettativa    : 2
  rabbia         : 2
  tristezza      : 1


## Analisi statistica: shift emotivo post → discussione (corpus completo)

Per ogni post nel corpus calcoliamo:
- **emozione del post** (metodo finale: ELIta α=0.5 + corpus_mean)
- **emozione della discussione** = emozione dominante del vettore medio dei commenti normalizzati

Poi costruiamo la **matrice di transizione** post\_emotion → discussion\_emotion.

In [10]:
# ── Parametri ─────────────────────────────────────────────────────────────────
MIN_COMMENTS = 2   # ignora post con meno di N commenti nel corpus

df_posts_only    = df_corpus[df_corpus['type'] == 'post']
df_comments_only = df_corpus[df_corpus['type'] == 'comment']

POSITIVE_EMOS = {'gioia', 'fiducia', 'sorpresa', 'aspettativa'}
NEGATIVE_EMOS = {'tristezza', 'rabbia', 'paura', 'disgusto'}

def polarita(emo):
    if emo in POSITIVE_EMOS: return 'positiva'
    if emo in NEGATIVE_EMOS: return 'negativa'
    return 'neutrale'

# ── Loop su tutti i post ───────────────────────────────────────────────────────
shift_rows = []

for _, post_row in df_posts_only.iterrows():
    pid = post_row['doc_id']

    sc_post_raw, found_post = score_single_document(pid, tok_dict, elita_idx, df_elita)
    if found_post == 0:
        continue  # post senza copertura ELIta → skip

    sc_post_n = normalizza(sc_post_raw, mu_e)
    emo_post  = emozione_dominante(sc_post_n)

    comms = df_comments_only[df_comments_only['post_id'] == pid]
    if len(comms) < MIN_COMMENTS:
        continue

    # Vettore medio dei commenti (solo quelli con copertura ELIta)
    comm_vecs = []
    for _, c_row in comms.iterrows():
        sc_c_raw, found_c = score_single_document(c_row['doc_id'], tok_dict, elita_idx, df_elita)
        if found_c == 0:
            continue
        comm_vecs.append(normalizza(sc_c_raw, mu_e))

    if not comm_vecs:
        continue

    mean_disc = {e: sum(v[e] for v in comm_vecs) / len(comm_vecs) for e in BASIC_EMOTIONS}
    emo_disc  = emozione_dominante(mean_disc)

    shift_rows.append({
        'post_id'       : pid,
        'n_commenti'    : len(comms),
        'n_comm_scored' : len(comm_vecs),
        'emo_post'      : emo_post,
        'emo_disc'      : emo_disc,
        'pol_post'      : polarita(emo_post),
        'pol_disc'      : polarita(emo_disc),
        'shift'         : emo_post != emo_disc,
        'shift_pol'     : polarita(emo_post) != polarita(emo_disc),
    })

df_shift = pd.DataFrame(shift_rows)

print(f'Post analizzati     : {len(df_shift)}  (≥{MIN_COMMENTS} commenti + copertura ELIta)')
print(f'Shift emozione      : {df_shift["shift"].sum()} ({df_shift["shift"].mean()*100:.1f}%)')
print(f'Shift polarità      : {df_shift["shift_pol"].sum()} ({df_shift["shift_pol"].mean()*100:.1f}%)')
print()
print('Emozione POST:        ', df_shift['emo_post'].value_counts().to_dict())
print('Emozione DISCUSSIONE: ', df_shift['emo_disc'].value_counts().to_dict())

Post analizzati     : 131  (≥2 commenti + copertura ELIta)
Shift emozione      : 99 (75.6%)
Shift polarità      : 51 (38.9%)

Emozione POST:         {'disgusto': 27, 'tristezza': 24, 'sorpresa': 19, 'gioia': 16, 'rabbia': 16, 'paura': 13, 'fiducia': 12, 'aspettativa': 4}
Emozione DISCUSSIONE:  {'gioia': 35, 'disgusto': 29, 'aspettativa': 21, 'fiducia': 13, 'sorpresa': 9, 'paura': 9, 'rabbia': 8, 'tristezza': 7}


In [11]:
# Matrice di transizione post_emotion → discussion_emotion

emos_order = [e for e in BASIC_EMOTIONS + ['neutrale']
              if e in df_shift['emo_post'].values or e in df_shift['emo_disc'].values]

counts_mat = pd.crosstab(df_shift['emo_post'], df_shift['emo_disc'])
counts_mat = counts_mat.reindex(
    index   =[e for e in emos_order if e in counts_mat.index],
    columns =[e for e in emos_order if e in counts_mat.columns],
    fill_value=0
)
pct_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0) * 100  # % per riga

# Heatmap
text_mat = [[f'{v:.0f}%' if v > 0 else '' for v in row] for row in pct_mat.values]
fig = go.Figure(go.Heatmap(
    z        = pct_mat.values,
    x        = list(pct_mat.columns),
    y        = list(pct_mat.index),
    colorscale='YlOrRd',
    text     = text_mat,
    texttemplate='%{text}',
    colorbar_title='%',
))
fig.update_layout(
    title      = 'Shift emotivo: emozione del post → emozione della discussione  (% per riga)',
    xaxis_title= 'Emozione discussione',
    yaxis_title= 'Emozione post',
    height     = 480,
)
fig.show()
print(pct_mat.round(1).to_string())

emo_disc     gioia  tristezza  rabbia  paura  disgusto  fiducia  sorpresa  aspettativa
emo_post                                                                              
gioia         43.8        0.0     6.2    0.0      12.5     12.5       6.2         18.8
tristezza     20.8        4.2    12.5   12.5      20.8      0.0       0.0         29.2
rabbia        25.0        0.0     6.2   12.5      25.0     18.8       0.0         12.5
paura          7.7        7.7     0.0   23.1      15.4     15.4       7.7         23.1
disgusto      14.8        7.4     3.7    3.7      44.4      7.4      14.8          3.7
fiducia       41.7        0.0     0.0    0.0       8.3     25.0       8.3         16.7
sorpresa      47.4       15.8    10.5    0.0      15.8      0.0      10.5          0.0
aspettativa    0.0        0.0     0.0    0.0       0.0     25.0       0.0         75.0


In [12]:
# ── Statistiche per polarità e shift principali ───────────────────────────────

# 1. Matrice polarità  positiva / negativa / neutrale
pol_mat = pd.crosstab(df_shift['pol_post'], df_shift['pol_disc'], margins=False)
pol_pct = pol_mat.div(pol_mat.sum(axis=1), axis=0) * 100
pol_order = [p for p in ['positiva','negativa','neutrale'] if p in pol_pct.index]
pol_pct = pol_pct.reindex(
    index  =[p for p in pol_order if p in pol_pct.index],
    columns=[p for p in pol_order if p in pol_pct.columns],
    fill_value=0
)

print('=== SHIFT DI POLARITÀ (post → discussione) ===')
print(pol_pct.round(1).to_string())
print()

# 2. Top-3 shift per ciascuna emozione del post
print('=== TOP SHIFT PER EMOZIONE DEL POST ===')
for emo in BASIC_EMOTIONS + ['neutrale']:
    sub = df_shift[df_shift['emo_post'] == emo]
    if len(sub) == 0:
        continue
    top  = sub['emo_disc'].value_counts()
    same = (sub['emo_disc'] == emo).sum()
    diff = len(sub) - same
    print(f'\n{emo.upper()}  ({len(sub)} post):')
    print(f'  stessa emozione: {same}/{len(sub)} ({same/len(sub)*100:.0f}%)')
    print(f'  shift:           {diff}/{len(sub)} ({diff/len(sub)*100:.0f}%)')
    top3 = '  '.join(f'{e}({cnt/len(sub)*100:.0f}%)' for e, cnt in top.head(3).items())
    print(f'  distribuzione → {top3}')

# 3. Heatmap polarità
fig2 = go.Figure(go.Heatmap(
    z         = pol_pct.values,
    x         = list(pol_pct.columns),
    y         = list(pol_pct.index),
    colorscale = 'Blues',
    text      = [[f'{v:.0f}%' for v in row] for row in pol_pct.values],
    texttemplate='%{text}',
    colorbar_title='%',
))
fig2.update_layout(
    title      = 'Shift di polarità: post → discussione  (% per riga)',
    xaxis_title= 'Polarità discussione',
    yaxis_title= 'Polarità post',
    height     = 350,
)
fig2.show()

=== SHIFT DI POLARITÀ (post → discussione) ===
pol_disc  positiva  negativa
pol_post                    
positiva      76.5      23.5
negativa      48.8      51.2

=== TOP SHIFT PER EMOZIONE DEL POST ===

GIOIA  (16 post):
  stessa emozione: 7/16 (44%)
  shift:           9/16 (56%)
  distribuzione → gioia(44%)  aspettativa(19%)  disgusto(12%)

TRISTEZZA  (24 post):
  stessa emozione: 1/24 (4%)
  shift:           23/24 (96%)
  distribuzione → aspettativa(29%)  gioia(21%)  disgusto(21%)

RABBIA  (16 post):
  stessa emozione: 1/16 (6%)
  shift:           15/16 (94%)
  distribuzione → disgusto(25%)  gioia(25%)  fiducia(19%)

PAURA  (13 post):
  stessa emozione: 3/13 (23%)
  shift:           10/13 (77%)
  distribuzione → paura(23%)  aspettativa(23%)  disgusto(15%)

DISGUSTO  (27 post):
  stessa emozione: 12/27 (44%)
  shift:           15/27 (56%)
  distribuzione → disgusto(44%)  sorpresa(15%)  gioia(15%)

FIDUCIA  (12 post):
  stessa emozione: 3/12 (25%)
  shift:           9/12 (75%)
  dist